 In order to first evaluate the bhdsgan I will use the following procedure:
 1. Sample from a truncated normal distribution with high variance --> gan should perform poorly
 2. Train gan on sampled data
 3. Generate new data and estimate density
 4. Compare actual and estimated density
 5. Sample from a truncated normal distribution with low variance --> gan should perform quite well
 6.Train gan on sampled data
 7. Generate new data and estimate density
 8. Compare actual and estimated density

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from torch import nn
from bhsgan import DiscriminatorBhsSim, GeneratorBhsSim,GeneratorBhsSimNormal
from dataset import SampleDataset
from ipmbhsgan import DiscriminatorIpmSim, GeneratorIpmSim
from trainer import (Trainer, TrainingParams, get_dis_loss_bhs,
                     get_dis_loss_ipm, get_dis_loss_wasserstein,
                     get_gen_loss_bhs, get_gen_loss_ipm,
                     get_gen_loss_wasserstein,get_dis_loss_kl, get_gen_loss_kl, get_dis_loss_rkl, get_gen_loss_rkl,
                     get_dis_loss_gan, get_gen_loss_gan, get_dis_loss_p, get_gen_loss_p)
from utils import get_device, get_noise, init_weights, plot_tensor_images, plot_losses, Positive, save_models_state_dict, load_model_state_dict, RevKlActivation, GanGanActivation
from wgan import DiscriminatorWassersteinSim, GeneratorWassersteinSim, GeneratorWassersteinSimNormal
import random

torch.set_default_dtype(torch.float64)
torch.manual_seed(96)
random.seed(96)

## sample from beta distribution

In [ ]:
# sample from beta distributions
mean_1_low = 0
sd_1_low = 1
mean_2_low = 5
sd_2_low = 1
mean_1_high = 0
sd_1_high = 4
mean_2_high = 5
sd_2_high = 4


In [ ]:
# beta_sample_high = np.random.beta(a_high, b_high, 10000)
normal_sample_1_high = np.random.choice(np.random.normal(mean_1_high, sd_1_high, 10000), 5000)
normal_sample_2_high = np.random.choice(np.random.normal(mean_2_high, sd_2_high, 10000), 5000)
sample_high = np.concatenate([normal_sample_1_high, normal_sample_2_high])
beta_sample_high = np.reshape(sample_high, (5000, 2))
sns.set_style('whitegrid')
sns.histplot(beta_sample_high)

In [ ]:
normal_sample_1_low = np.random.choice(np.random.normal(mean_1_low, sd_1_low, 10000), 5000)
normal_sample_2_low = np.random.choice(np.random.normal(mean_2_low, sd_2_low, 10000), 5000)
sample_low = np.concatenate([normal_sample_1_low, normal_sample_2_low])
beta_sample_low = np.reshape(sample_low, (5000, 2))
sns.set_style('whitegrid')
sns.histplot(beta_sample_low)

In [ ]:
training_set_high = SampleDataset(beta_sample_high)
training_set_low = SampleDataset(beta_sample_low)

In [ ]:
test_noise = get_noise(10000, 2)

## Train BHS GAN

In [ ]:
training_params = TrainingParams(lr_dis=0.0002, lr_gen=0.0002, num_epochs=15, num_dis_updates=4, num_gen_updates = 3, beta_1=0.5, batch_size=128)
 
# get device to train on
device = "cpu"

In [ ]:
# Create the dataloaders
dataloader_high = torch.utils.data.DataLoader(training_set_high, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
dataloader_low = torch.utils.data.DataLoader(training_set_low, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)

In [ ]:
# Create the dataloaders
dataloader_high = torch.utils.data.DataLoader(training_set_high, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
dataloader_low = torch.utils.data.DataLoader(training_set_low, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
# initialize nets
final_activation = Positive
generator_high = GeneratorBhsSimNormal()
discriminator_high = DiscriminatorBhsSim(final_activation)
generator_low = GeneratorBhsSimNormal()
discriminator_low = DiscriminatorBhsSim(final_activation)

In [ ]:
# init Trainer
trainer_high = Trainer(training_params, generator_high, discriminator_high)
trainer_low = Trainer(training_params, generator_low, discriminator_low)

### Train on high variance samples

In [ ]:
# training loop
trained_bhs_high = trainer_high.train_gan(dataloader_high, get_dis_loss_bhs, get_gen_loss_bhs, False, print_intermediate=False)

In [ ]:
generated_data = trained_bhs_high.generator(test_noise)
generated_sample = torch.reshape(generated_data, (1, 20000)).detach().numpy().ravel()

In [ ]:
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample, bins=50)

### Train on low variance samples

In [ ]:
# training loop
trained_bhs_low = trainer_low.train_gan(dataloader_low, get_dis_loss_bhs, get_gen_loss_bhs, False, print_intermediate=False)

In [ ]:
generated_data_low = trained_bhs_low.generator(test_noise)
generated_sample_low = torch.reshape(generated_data_low, (1, 20000)).detach().numpy().ravel()

In [ ]:
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_low, bins=50)

## Wasserstein GAN 

### High Variance

In [ ]:
dataloader_wasserstein_high = torch.utils.data.DataLoader(training_set_high, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
generator_wasserstein_high = GeneratorWassersteinSimNormal()
discriminator_wasserstein_high = DiscriminatorWassersteinSim()
trainer_wgan_high = Trainer(training_params, generator_wasserstein_high, discriminator_wasserstein_high)

In [ ]:
# training loop
trained_wgan_high = trainer_wgan_high.train_gan(dataloader_wasserstein_high, get_dis_loss_wasserstein, get_gen_loss_wasserstein, True, print_intermediate=False)

In [ ]:
generated_data_wasserstein_high = trained_wgan_high.generator(test_noise)
generated_sample_wasserstein_high = torch.reshape(generated_data_wasserstein_high, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_wasserstein_high, bins=100)

### Low Variance

In [ ]:
dataloader_wasserstein_low = torch.utils.data.DataLoader(training_set_low, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
generator_wasserstein_low = GeneratorWassersteinSimNormal()
discriminator_wasserstein_low = DiscriminatorWassersteinSim()
trainer_wgan_low = Trainer(training_params, generator_wasserstein_low, discriminator_wasserstein_low)

In [ ]:
# training loop
trained_wgan_low = trainer_wgan_low.train_gan(dataloader_wasserstein_low, get_dis_loss_wasserstein, get_gen_loss_wasserstein, True, print_intermediate=False)

In [ ]:
generated_data_wasserstein_low = trained_wgan_low.generator(test_noise)
generated_sample_wasserstein_low = torch.reshape(generated_data_wasserstein_low, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_wasserstein_low, bins=50)

## GAN GAN 

In [ ]:
# Create the dataloaders
dataloader_high = torch.utils.data.DataLoader(training_set_high, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
dataloader_low = torch.utils.data.DataLoader(training_set_low, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
# initialize nets
final_activation = nn.Sigmoid
generator_gan_high = GeneratorBhsSimNormal()
discriminator_gan_high = DiscriminatorBhsSim(final_activation)
generator_gan_low = GeneratorBhsSimNormal()
discriminator_gan_low = DiscriminatorBhsSim(final_activation)

In [ ]:
### High Variance

In [ ]:
trainer_gan_high = Trainer(training_params, generator_gan_high, discriminator_gan_high)

In [ ]:
# training loop
trained_gan_high = trainer_gan_high.train_gan(dataloader_high, get_dis_loss_gan, get_gen_loss_gan, False, print_intermediate=False)

In [ ]:
generated_data_gan_high = trained_gan_high.generator(test_noise)
generated_sample_gan_high = torch.reshape(generated_data_gan_high, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_gan_high, bins=50)

In [ ]:
### Low Variance

In [ ]:
trainer_gan_low = Trainer(training_params, generator_gan_low, discriminator_gan_low)

In [ ]:
# training loop
trained_gan_low = trainer_gan_low.train_gan(dataloader_low, get_dis_loss_gan, get_gen_loss_gan, False, print_intermediate=False)

In [ ]:
generated_data_gan_low = trained_gan_low.generator(test_noise)
generated_sample_gan_low = torch.reshape(generated_data_gan_low, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_gan_low, bins=50)

## Pearson GAN

In [ ]:
# Create the dataloaders
dataloader_high = torch.utils.data.DataLoader(training_set_high, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
dataloader_low = torch.utils.data.DataLoader(training_set_low, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
# initialize nets
final_activation = nn.Identity
generator_pearson_high = GeneratorBhsSimNormal()
discriminator_pearson_high = DiscriminatorBhsSim(final_activation)
generator_pearson_low = GeneratorBhsSimNormal()
discriminator_pearson_low = DiscriminatorBhsSim(final_activation)

In [ ]:
### High Variance

In [ ]:
trainer_pearson_high = Trainer(training_params, generator_pearson_high, discriminator_pearson_high)

In [ ]:
# training loop
trained_pearson_high = trainer_pearson_high.train_gan(dataloader_high, get_dis_loss_p, get_gen_loss_p, False, print_intermediate=False)

In [ ]:
generated_data_pearson_high = trained_pearson_high.generator(test_noise)
generated_sample_pearson_high = torch.reshape(generated_data_pearson_high, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_pearson_high, bins=50)

In [ ]:
### Low Variance

In [ ]:
trainer_pearson_low = Trainer(training_params, generator_pearson_low, discriminator_pearson_low)

In [ ]:
# training loop
trained_pearson_low = trainer_pearson_low.train_gan(dataloader_low, get_dis_loss_p, get_gen_loss_p, False, print_intermediate=False)

In [ ]:
generated_data_pearson_low = trained_pearson_low.generator(test_noise)
generated_sample_pearson_low = torch.reshape(generated_data_pearson_low, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_pearson_low, bins=50)

## KL GAN

In [ ]:
# Create the dataloaders
dataloader_high = torch.utils.data.DataLoader(training_set_high, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
dataloader_low = torch.utils.data.DataLoader(training_set_low, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
# initialize nets
final_activation = nn.Identity
generator_kl_high = GeneratorBhsSimNormal()
discriminator_kl_high = DiscriminatorBhsSim(final_activation)
generator_kl_low = GeneratorBhsSimNormal()
discriminator_kl_low = DiscriminatorBhsSim(final_activation)

In [ ]:
### High Variance

In [ ]:
trainer_kl_high = Trainer(training_params, generator_kl_high, discriminator_kl_high)

In [ ]:
# training loop
trained_kl_high = trainer_kl_high.train_gan(dataloader_high, get_dis_loss_kl, get_gen_loss_kl, False, print_intermediate=False)

In [ ]:
generated_data_kl_high = trained_kl_high.generator(test_noise)
generated_sample_kl_high = torch.reshape(generated_data_kl_high, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_kl_high, bins=50)

In [ ]:
### Low Variance

In [ ]:
trainer_kl_low = Trainer(training_params, generator_kl_low, discriminator_kl_low)

In [ ]:
# training loop
trained_kl_low = trainer_kl_low.train_gan(dataloader_low, get_dis_loss_kl, get_gen_loss_kl, False, print_intermediate=False)

In [ ]:
generated_data_kl_low = trained_kl_low.generator(test_noise)
generated_sample_kl_low = torch.reshape(generated_data_kl_low, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_kl_low, bins=50)

## RV KL GAN

In [ ]:
# Create the dataloaders
dataloader_high = torch.utils.data.DataLoader(training_set_high, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
dataloader_low = torch.utils.data.DataLoader(training_set_low, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
# initialize nets
final_activation = RevKlActivation
generator_rvkl_high = GeneratorBhsSimNormal()
discriminator_rvkl_high = DiscriminatorBhsSim(final_activation)
generator_rvkl_low = GeneratorBhsSimNormal()
discriminator_rvkl_low = DiscriminatorBhsSim(final_activation)

In [ ]:
### High Variance

In [ ]:
trainer_rvkl_high = Trainer(training_params, generator_rvkl_high, discriminator_rvkl_high)

In [ ]:
# training loop
trained_rvkl_high = trainer_rvkl_high.train_gan(dataloader_high, get_dis_loss_rkl, get_gen_loss_rkl, False, print_intermediate=False)

In [ ]:
generated_data_rvkl_high = trained_rvkl_high.generator(test_noise)
generated_sample_rvkl_high = torch.reshape(generated_data_rvkl_high, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_rvkl_high, bins=50)

In [ ]:
### Low Variance

In [ ]:
trainer_rvkl_low = Trainer(training_params, generator_rvkl_low, discriminator_rvkl_low)

In [ ]:
# training loop
trained_rvkl_low = trainer_rvkl_low.train_gan(dataloader_low, get_dis_loss_rkl, get_gen_loss_rkl, False, print_intermediate=False)

In [ ]:
generated_data_rvkl_low = trained_rvkl_low.generator(test_noise)
generated_sample_rvkl_low = torch.reshape(generated_data_rvkl_low, (1, 20000)).detach().numpy().ravel()
# plot resulting density
sns.set_style('whitegrid')
sns.histplot(generated_sample_rvkl_low, bins=50)

## Now I train the IPM Version for the BHS GAN

In [ ]:
# initialize nets
generator_ipm = GeneratorIpmSim()
discriminator_ipm = DiscriminatorIpmSim()

In [ ]:
dataloader_ipm = torch.utils.data.DataLoader(training_set_low, batch_size=training_params.batch_size,
                                         shuffle=True, num_workers=1)
# init Trainer
trainer_ipm_gan = Trainer(training_params, generator_ipm, discriminator_ipm)

In [ ]:
# training loop
trained_ipm_gan = trainer_ipm_gan.train_gan(dataloader_ipm, get_dis_loss_ipm, get_gen_loss_ipm, False)

In [ ]:
generated_data_ipm = trained_ipm_gan.generator(test_noise)
generated_sample_ipm = torch.reshape(generated_data_ipm, (1, 1000)).detach().numpy().ravel()

In [ ]:
# plot resulting density
sns.set_style('whitegrid')
sns.kdeplot(generated_sample_ipm, bw=0.5)

In [ ]:
# plot losses
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(trained_ipm_gan.generator_losses,label="G-Loss")
plt.plot(trained_ipm_gan.discriminator_losses,label="D-Loss")
plt.xlabel("iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# check mean and varaince of generated data
np.mean(generated_sample_ipm)

In [ ]:
np.std(generated_sample_ipm)

In [ ]:
data_points = np.linspace(-1, 2, 100)
def conjugate(points):
    conjugate_1 = 2 * (-1 + np.sqrt(1 + points)) * np.exp(-1 + np.sqrt(1 + points)) 
    conjugate_2 = 2 * (-1 - np.sqrt(1 + points)) * np.exp(-1 - np.sqrt(1 + points)) 
    return np.where(points >= 0, conjugate_1, conjugate_2)
plt.plot(data_points, conjugate(data_points))

In [ ]:
def f(points):
    f_1 = points * np.log(points)**2
    f_2 = -points * np.log(points)**2
    return np.where(points >= np.exp(-1), f_1, f_2)
plt.plot(data_points, f(data_points))

In [ ]:
2 * (-1 - np.sqrt(1 + -1)) * np.exp(-1 - np.sqrt(1 + -1))

In [ ]:
conjugate(data_points)